### Carga de Email


- Conexion a casilla de correo
- Se listan todos los correos
- Se selecciona el máximo correo de la tabla actual de correos
- Se cargan los correos
- Se agrega la tabla de control
- secretos

In [0]:
from datetime import datetime
import time 

status = 0
status_ejecucion = 0
desc_status_ejecucion = "[OK]"

starttime_nifi = datetime.now().strftime("%Y%m%d %H%M%S")

In [0]:
dir_adls            = dbutils.widgets.text("dir_adls","abfss://bigdatadev@stbigdatadev02.dfs.core.windows.net/data/interacciones/casos/isc/respuesta_email")
pipelineRunId       = dbutils.widgets.text("pipelineRunId", "test")
catalog_control     = dbutils.widgets.text("catalog_control", "biproduccion.control.control_ingestas")
dataType            = dbutils.widgets.text("dataType" , "raw")
plataforma          = dbutils.widgets.text("plataforma", "fareco")
categoria           = dbutils.widgets.text("categoria", "respuestas_email")
nomb_proc           = dbutils.widgets.text("nomb_proc", "raw_interacciones.respuestas_email")
catalog_name        = dbutils.widgets.text("catalog_name", "bi_ingestas")

In [0]:
try:
    dir_adls            = dbutils.widgets.get("dir_adls")
    pipelineRunId       = dbutils.widgets.get("pipelineRunId")
    catalog_control     = dbutils.widgets.get("catalog_control")
    dataType            = dbutils.widgets.get("dataType")
    plataforma          = dbutils.widgets.get("plataforma")
    categoria           = dbutils.widgets.get("categoria")
    loadType            = "i"
    periodicity         = "d"
    nomb_proc           = dbutils.widgets.get("nomb_proc")
    catalog_name        = dbutils.widgets.get("catalog_name")

    output_table        = catalog_name + "." + nomb_proc
    path_data           = dir_adls + "/" + dataType

except Exception as e:
    status_ejecucion = 1
    desc_status_ejecucion = "[ERROR] " + str(e)
    print("[ERROR] " + str(e))

### Conexión a Casilla de Correos IMAP

In [0]:

# Establecer conexion segura con el servidor IMAP
import imaplib
import ssl

from email.header import decode_header
import email

import os
import re
from email.utils import parsedate_to_datetime
import base64

N = 100 # cantidad de correos a leer en caso de que no exista UID disponible en abla
CHUNK_SIZE = 50  # cantidad de archivos por lote

# Credenciales: TODO: Usar Secretos
EMAIL_ACCOUNT = "feedback2@movistar.cl"
PASSWORD = dbutils.secrets.get(scope="secrets-ingestas", key="sec-email-feedback2-movistar-cl")


try:
    # Create an SSL context with specific protocol and cipher settings
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.set_ciphers('DEFAULT:@SECLEVEL=1')  # Lower security level if needed for compatibility
    ctx.check_hostname = False  # Disable hostname checking
    ctx.verify_mode = ssl.CERT_NONE  # Disable certificate verification

    # Connect to the IMAP server using the custom SSL context
    mail = imaplib.IMAP4_SSL('imap.tie.cl', 993, ssl_context=ctx)
    mail.login(EMAIL_ACCOUNT, PASSWORD)
    mail.list()
    mail.select('inbox')
    # "data" es una lista de IDs de mensajes en la bandeja de entrada
    result, data = mail.uid('search', None, "ALL") # (ALL/UNSEEN)
except Exception as e:
    status_ejecucion = 1
    desc_status_ejecucion = "[ERROR] " + str(e)
    print("[ERROR] " + str(e))

In [0]:
# Obtener todos los UIDs ordenados de mayor a menor (mas reciente a menos reciente)
if status_ejecucion == 0:
    uids = [int(s) for s in data[0].split()]
    uids.sort()

In [0]:
# crear la tabla temporal en desarrollo
ddl_query = f"""
CREATE TABLE IF NOT EXISTS {output_table} (
    financial_account_key STRING,
    fecha_respuesta_correo timestamp,
    texto_correo_encoded STRING,
    telefono STRING,
    id_correo long,
    bigdata_close_date date,
    bigdata_ctrl_id long
)
USING DELTA
LOCATION '{path_data}'
"""

spark.sql(ddl_query)

In [0]:
# Obtener el máximo ID de correo en la tabla actual
try: 
    query = f"SELECT MAX(id_correo) AS max_id_correo FROM {output_table}"
    max_id_correo_tabla = spark.sql(query).collect()[0]['max_id_correo']
except Exception as e:
    max_id_correo_tabla = None
    print(f"[ERROR] {e}")

print(f"[INFO] El maximo ID de correo en la tabla es {max_id_correo_tabla}")

In [0]:
# En base al máximo ID de correo en la tabla, se obtiene el UID mínimo a partir del cual se leerán los correos
# En caso de no existir la tabla, se traerán los N últimos correos
max_uid = int(uids[-1])  # Último UID
if max_id_correo_tabla is None:
    min_uid = max_uid - N
else: 
    min_uid = int(max_id_correo_tabla)

# De la lista de UIDs, se obtiene el índice del UID mínimo dado el "min_uid" calculado. 
# Esto se hace ya que no necesariamente UIDS es una lista consecutiva de correos. 
# Puede haber un UID 200 y el siguiente es el 202 y no el 201

try:
    index = uids.index(min_uid) + 1 if min_uid is not None else 0
except ValueError:
    index = 0 # Cuando esto ocurra, comenzará a buscar "todos" los correos, desde el uid mas pequeño de uids

print(f"[INFO] Se buscarán {max_uid - min_uid} correos")

In [0]:
# Listas que almacenarán los datos durante la descarga de correos en la iteración
uid_list = []
asunto_list = []
fecha_list = []
mensaje_correo_list = []

try:
    # Se itera sobre los UIDs por el tamaño de lote
    # Al hacerlo por lotes se usa una conexión para traer todos los correos en el lote y no 
    # uno a la vez
    for i in range(index, len(uids), CHUNK_SIZE):
        chunk = uids[i:i+CHUNK_SIZE]
        min_uid = chunk[0]
        max_uid = chunk[-1]
        
        uid_range = f"{min_uid}:{max_uid}"
        # '(UID BODY.PEEK[])' es la cláusula que búsca los correos sin abrirlos en la casilla
        # Esto es útil para que no se marque como leído el correo en la bandeja de entrada.
        result, data = mail.uid('fetch', uid_range, '(UID BODY.PEEK[])')
        
        # `data` contiene pares (b'N UID FETCH', b'contenido') + un cierre con `None`
        for item in data:
            if isinstance(item, tuple) and len(item) == 2:
                raw_header = item[0].decode()
                match = re.search(r'UID (\d+)', raw_header)
                if not match:
                    continue
                uid = int(match.group(1))
                uid_list.append(uid)

                raw_email = item[1]
                try:
                    msg = email.message_from_bytes(raw_email)

                    # Obtener asunto
                    subject = msg.get("Subject", "")
                    decoded_subject, encoding = decode_header(subject)[0]
                    if isinstance(decoded_subject, bytes):
                        subject = decoded_subject.decode(encoding if encoding else "utf-8")

                except Exception:
                    subject = "(sin asunto)"
                asunto_list.append(subject)

                # Obtener fecha, independientemente del error de asunto
                try:
                    date_str = msg.get("Date")
                    if date_str:
                        date_obj = parsedate_to_datetime(date_str)
                        fecha_list.append(date_obj)
                    else:
                        fecha_list.append(None)
                except Exception:
                    fecha_list.append(None)

                try:
                    # Convertir el mensaje a bytes
                    raw_bytes = msg.as_bytes()
                    # Codificar en base64
                    encoded_msg = base64.b64encode(raw_bytes).decode("utf-8")
                    
                except Exception as e:
                    print(f"Error al guardar el mensaje UID {uid}: {e}")
                mensaje_correo_list.append(encoded_msg)

    # Se cierra conexión con casilla de correo
    print("[INFO] Cerrando conexión")
    mail.logout()

except Exception as e:
    status_ejecucion = 1
    desc_status_ejecucion = "[ERROR] " + str(e)
    print("[ERROR] " + str(e))

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import regexp_extract, col, current_date, lit, from_unixtime, unix_timestamp
from datetime import datetime

# Preparar datos como lista de diccionarios
data = [
    {
        'id_correo': uid,
        'fecha_respuesta_correo': fecha,
        'texto_correo_encoded': mensaje,
        'asunto': asunto
    }
    for uid, fecha, mensaje, asunto in zip(uid_list, fecha_list, mensaje_correo_list, asunto_list)
]

# Crear Spark DataFrame directamente con los datos de la lista y se distribuye a 4 nodos
spark_df = spark.createDataFrame(data).repartition(4)

# Timestamp
formatter_id = datetime.now().strftime("%Y%m%d%H%M%S")
bigdata_ctrl_id = f"{formatter_id}{1:03d}"

# Agregar columnas
spark_df = (
    spark_df
    .withColumn("financial_account_key", regexp_extract(col("asunto"), r"Cliente:\s*(\d+)", 1))
    .withColumn("telefono", regexp_extract(col("asunto"), r"(?:Numero(?: Celular)?:)\s*(\d+)", 1))
    .withColumn("bigdata_close_date", current_date())
    .withColumn("bigdata_ctrl_id", lit(bigdata_ctrl_id).cast("long"))
    .drop("asunto")
)


In [0]:
spark_df.printSchema()

In [0]:
original_row_count = str(spark_df.count())
final_row_count = original_row_count
dif_row_count = "0"

In [0]:
# Se guardan los datos en tabla de salida
spark_df.write.mode("append").saveAsTable(output_table)

## Tabla de Control

In [0]:
subida_datos = "CREATE EXTERNAL TABLE IF NOT EXISTS raw_interacciones.respuestas_email"

nomb_proc = subida_datos.split('\n')[0].lower().split("create external table if not exists ")[1].strip()
print("[INFO] nombre proceso " + nomb_proc)

# SE CONCATENA SPARK AL NOMBRE DEL PROCESO
process_name = "spark_" + nomb_proc
print("[INFO] spark + nombre proceso " + process_name)

In [0]:
from datetime import datetime

# SE CUENTA LA CANTIDAD DE ARCHIVOS GENERADOS
final_number_of_files = ""

# SE GUARDA EL NOMBRE FINAL DEL ARCHIVO
current = datetime.now().strftime("%Y%m%d%H%M%S")
end_file_name = f"{dataType}.{plataforma}.{categoria}.{loadType}.{periodicity}.{current}.snappy.parquet"

# SE CALCULA EL TIMESTAMP
current_ins = datetime.now()

# SE CREA UN FORMATO DE FECHA Y SE FORMATEA EL TIMESTAMP
insert_data_ctrl_date = current_ins.strftime("%Y-%m-%d %H:%M:%S")

data_source_type = "email"

In [0]:
from datetime import datetime

endtime_nifi              = datetime.now().strftime("%Y%m%d %H%M%S")
format_actual             = "%Y%m%d %H%M%S"
starttime_nifi_dateFormat = datetime.strptime(starttime_nifi, format_actual)
endtime_nifi_dateFormat   = datetime.strptime(endtime_nifi, format_actual)
totaltime_nifi            = str((endtime_nifi_dateFormat - starttime_nifi_dateFormat).total_seconds())
starttime_spark = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("starttime " + starttime_spark)

In [0]:
# Variables que no tienen contenido en framework original
process_type    = ""
final_file_size = "0"
filename        = "CasillaFeedback"
original_file_date = ""
original_file_size = ""

In [0]:
print(status_ejecucion, desc_status_ejecucion, bigdata_ctrl_id, process_name, data_source_type, filename, original_file_date, starttime_nifi, endtime_nifi, totaltime_nifi, starttime_spark, process_type, original_file_size, final_file_size, original_row_count, final_row_count, dif_row_count, final_number_of_files, end_file_name, insert_data_ctrl_date, path_data, pipelineRunId)

In [0]:
# SE AGREGAN LOS PARAMETROS DE CONTROL A LA LISTA
control_data = [(str(status_ejecucion), desc_status_ejecucion, bigdata_ctrl_id, process_name, data_source_type, filename, original_file_date, starttime_nifi, endtime_nifi, totaltime_nifi, starttime_spark, process_type, original_file_size, final_file_size, original_row_count, final_row_count, dif_row_count, final_number_of_files, end_file_name, insert_data_ctrl_date, path_data, pipelineRunId)]

In [0]:
control_data

In [0]:
columns = [
    "status", "desc_status", "big_data_ctrl_id", "process_name", "data_source_type", "data_source_name",
    "original_file_date", "starttime_nifi", "endtime_nifi", "totaltime_nifi", "starttime_spark",
    "process_type", "original_file_size", "final_file_size", "original_row_count", "final_row_count",
    "dif_row_count", "final_number_of_files", "end_file_name", "insert_data_ctrl_date", "hdfs_path",
    "pipelineRunId"
]

control_dataframe = spark.createDataFrame(control_data, columns)

# Obtener el timestamp actual (end)
end = int(time.time() * 1000)  # milisegundos

# Formatear el timestamp
endtime_spark = datetime.fromtimestamp(end / 1000).strftime("%Y-%m-%d %H:%M:%S")

# Calcular duración del proceso Spark
totaltime_spark = (end - int(current)) / 1000  # asumiendo que 'current' está definido en milisegundos

# Calcular duración total del proceso (Spark + NiFi)
totaltime_process = totaltime_spark + int(float(totaltime_nifi))

# Agregar columnas al DataFrame y transformar formatos
control_dataframe = (
    control_dataframe
    .withColumn("endtime_spark", lit(endtime_spark))
    .withColumn("totaltime_spark", lit(totaltime_spark).cast("float"))
    .withColumn("totaltime_process", lit(totaltime_process).cast("float"))
    .withColumn("original_file_date", from_unixtime(unix_timestamp("original_file_date", "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("starttime_nifi", from_unixtime(unix_timestamp("starttime_nifi", "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("endtime_nifi", from_unixtime(unix_timestamp("endtime_nifi", "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
    .select(
        "status", "desc_status", "big_data_ctrl_id", "process_name", "data_source_type", "data_source_name",
        "original_file_date", "starttime_nifi", "endtime_nifi", "totaltime_nifi", "starttime_spark",
        "endtime_spark", "totaltime_spark", "totaltime_process", "insert_data_ctrl_date", "process_type",
        "original_file_size", "final_file_size", "original_row_count", "final_row_count", "dif_row_count",
        "final_number_of_files", "end_file_name", "hdfs_path", "pipelineRunId"
    )
)


In [0]:
display(control_dataframe)

In [0]:
# Guardar el DataFrame en Hive (Unity Catalog o Hive clásico)
control_dataframe.write.mode("append").option("mergeSchema", "true").saveAsTable(catalog_control)

In [0]:
if status_ejecucion == 0:
    dbutils.notebook.exit("OK")
else:
    raise Exception("Error en el Try-Catch del proceso.")